# MATE 1000: Worker 5 of 5 (positions [820:1000])

Part of a 5-way parallel extension of the verified 100-position thinking-mode run to the full 1000-position mate-selection-test.json (5, not 6: Kaggle caps concurrent CPU kernel sessions at 5 per account). Each worker owns a disjoint 180-position slice and its own API key (the gateway serializes per-key, so 5 keys = real 5x parallelism, not 5 queued requests on one key).

Same methodology as the archived 100 (HF run `2026-08-04T16:09:37Z`, 94/100): thinking ON, no `--thinking-budget` cap, `--force-answer-prompt`, `--max_new_tokens 131072`.

Secrets needed on **this specific kernel**: `GITHUB_TOKEN`, `HF_WRITE_TOKEN` (or `HF_TOKEN`) -- same secrets already used elsewhere, just attach them here too -- plus a NEW secret named `OPENCODE_API_KEY_5` holding this worker's own key. Notebook editor -> + Add -> Add secret -> Save -> Kernel -> Restart & Run All.

## 1. Secrets

In [ ]:
import os

def _secret(names):
    for name in names:
        if os.environ.get(name):
            return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                val = client.get_secret(name)
                if val:
                    return val
            except Exception:
                continue
    except Exception:
        pass
    return None

github_token = _secret(["GITHUB_TOKEN", "GH_TOKEN"])
hf_token = _secret(["HF_WRITE_TOKEN", "HF_TOKEN"])
opencode_key = _secret(['OPENCODE_API_KEY_5'])

missing = [n for n, v in [("GITHUB_TOKEN", github_token), ("HF_WRITE_TOKEN/HF_TOKEN", hf_token),
                          ('OPENCODE_API_KEY_5', opencode_key)] if not v]
if missing:
    raise RuntimeError(
        f"missing Kaggle secrets: {missing}. Attach GITHUB_TOKEN and "
        "HF_WRITE_TOKEN (or HF_TOKEN) -- the same secrets already used on "
        f"other kernels, just attach them here too -- plus create+attach a "
        f"NEW secret named 'OPENCODE_API_KEY_5' holding worker 5's "
        "own key (see the local .env: OPENCODE_API_KEY_5). Then: "
        "Notebook editor -> + Add -> Add secret -> Save -> Kernel -> Restart & Run All."
    )
os.environ["GITHUB_TOKEN"] = github_token
os.environ["HF_WRITE_TOKEN"] = hf_token
os.environ["OPENCODE_API_KEY"] = opencode_key
print("secrets resolved: GITHUB_TOKEN, HF_WRITE_TOKEN, OPENCODE_API_KEY (worker 5)")

## 2. Get the repo (main branch, explicitly)

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "chess-slm-benchmark"
if REPO.exists():
    shutil.rmtree(REPO)

url = "https://github.com/Vedang-P/chess-slm-benchmark.git"
url = url.replace("https://", f"https://x-access-token:{os.environ['GITHUB_TOKEN']}@")
res = subprocess.run(["git", "clone", "--quiet", "-b", "main", url, str(REPO)],
                     capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError("clone failed (bad/missing GITHUB_TOKEN?): " + res.stderr[-300:])
os.chdir(REPO)
print("cwd:", Path.cwd())

## 3. Dependencies (CPU only -- no torch/gpu install needed for the gateway arm)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-U", "-r", "requirements.txt"], check=True)
print("deps installed")

## 4. Engine/dataset gate

In [ ]:
status = subprocess.run([sys.executable, "scripts/test_engine.py"], capture_output=True, text=True)
print(status.stdout[-3000:])
if status.returncode != 0:
    print(status.stderr[-2000:])
    raise RuntimeError("test_engine failed -- see output above")
print("ALL TESTS PASSED")

## 5. Recover this worker's own progress (if this is a relaunch after a died/timed-out session)

In [ ]:
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

OUT_DIR = Path('results/mate-1000-w5')
OUT_DIR.mkdir(parents=True, exist_ok=True)
BENCH_RUN_ID = 'mate1000-w5'
RUN_NAME = 'deepseek-v4-flash_mate-selection-test_strategy'

# /kaggle/working is wiped on every "Restart & Run All" (the clone cell
# re-creates it), so if THIS kernel died mid-slice and is being relaunched,
# there is nothing local to --resume from unless we pull this worker's own
# progress back down first. Each worker uploads to its OWN run_id path
# (see run_mate_eval.py --hf-upload-every), so this can never collide with
# another worker's or the original 100's data.
try:
    src = hf_hub_download(
        repo_id="vedangfake/chess-bench-results", repo_type="dataset",
        filename=f"runs/{BENCH_RUN_ID}/{RUN_NAME}.samples.jsonl",
        token=os.environ.get("HF_WRITE_TOKEN") or os.environ.get("HF_TOKEN"))
    dest = OUT_DIR / f"{RUN_NAME}.samples.jsonl"
    dest.write_bytes(Path(src).read_bytes())
    n = sum(1 for line in dest.read_text().splitlines() if line.strip())
    print(f"recovered {n} previously-scored positions for this worker from HF -- --resume will skip them")
except Exception as e:
    print(f"nothing to recover (first launch of this worker, or no checkpoint yet): {e}")

## 6. Run positions [820:1000]

Uploads to HF every 25 positions (not just at the end) and publishes live state to `monitor/workers/w5.state.json` on the public monitor repo -- scripts/aggregate_live_state.py combines all 5 workers + the completed 100 into the canonical dashboard the website reads.

In [ ]:
import time
from pathlib import Path

os.environ["BENCH_RUN_ID"] = 'mate1000-w5'
out = Path('results/mate-1000-w5')
out.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, "scripts/run_mate_eval.py",
       "--model", "deepseek-v4-flash",
       "--offset", "820",
       "--n", "180",
       "--max_new_tokens", "131072",
       "--force-answer-prompt",
       "--worker-id", 'w5',
       "--output_dir", 'results/mate-1000-w5',
       "--live-push",
       "--resume",
       "--verbose"]
print("running:", " ".join(cmd))
t0 = time.time()
# unbuffered + merged stderr so a crash's real traceback lands in THIS
# cell's own output instead of vanishing into the subprocess's own stderr
# pipe (which showed up nowhere -- an earlier version of this cell just
# printed a return code and let the notebook finish "successfully" even
# when the actual run crashed instantly).
res = subprocess.run(cmd, stderr=subprocess.STDOUT)
elapsed_h = (time.time() - t0) / 3600
print(f"run exited rc={res.returncode} after {elapsed_h:.2f}h")
if res.returncode != 0:
    raise RuntimeError(
        f"run_mate_eval.py exited rc={res.returncode} after only "
        f"{elapsed_h:.2f}h -- this is a real failure, not a completed run. "
        "See the subprocess output printed above this cell for the actual "
        "traceback."
    )

## 7. This worker's summary

In [ ]:
import json, glob, collections

rows = []
for f in glob.glob(str(Path('results/mate-1000-w5') / "*.samples.jsonl")):
    for line in open(f):
        if line.strip(): rows.append(json.loads(line))
n = len(rows)
by_status = collections.Counter(r["status"] for r in rows)
correct = sum(bool(r["compliance"]) for r in rows if r["status"] != "api_error")
scored = n - by_status.get("api_error", 0)
print(f"this worker: {n} / 180 positions attempted")
print("status breakdown:", dict(by_status))
if scored:
    print(f"accuracy (of {scored} scored): {correct}/{scored} = {correct/scored:.3f}")

## Notes
- Scope: ONLY positions [820:1000) of mate-selection-test.json. Never touches the first 100 or another worker's slice.
- If this session dies/times out, just push this same notebook again (or Restart & Run All): step 5 recovers this worker's own progress from HF, and --resume skips it.
- Results upload to HF under run_id `mate1000-w5` and the live dashboard at chess-bench-live.pages.dev.